In [3]:
from pathlib import Path
import pandas as pd

results_path = Path().resolve().parent / "experiments"
models = [
    "L1-Qwen3-8B-Max",
    "Qwen3-8B",
    "L1-Qwen-1.5B-Exact",
    "TokenSkip-Qwen2",
    "QwQ-32B-thinkprune-iter2k",
    "LCR1_7B",
]

datasets = ["math-500", "gsm8k", "olympiad", "amc", "aime-250"]

rows = []
for model in models:
    model_dir = results_path / model
    for dataset in datasets:
        parquet_file = model_dir / f"{dataset}_results.parquet"
        if not parquet_file.exists():
            continue
        df = pd.read_parquet(parquet_file)
        total = len(df)
        correct = df["is_correct"].sum()
        accuracy = correct / total if total > 0 else 0.0
        avg_tokens = df["token_count"].mean()
        rows.append({
            "model": model,
            "dataset": dataset,
            "accuracy": accuracy,
            "num_correct": correct,
            "num_total": total,
            "avg_tokens": avg_tokens,
        })

summary = pd.DataFrame(rows)
summary

,model,dataset,accuracy,num_correct,num_total,avg_tokens
0,L1-Qwen3-8B-Max,math-500,0.731463,365,499,2429.765531
1,L1-Qwen3-8B-Max,gsm8k,0.805914,1063,1319,2137.841547
2,L1-Qwen3-8B-Max,olympiad,0.344214,232,674,3119.321958
3,L1-Qwen3-8B-Max,amc,0.725000,29,40,2749.200000
4,L1-Qwen3-8B-Max,aime-250,0.380000,95,250,3406.496000
5,Qwen3-8B,math-500,0.757515,378,499,5895.448898
6,Qwen3-8B,gsm8k,0.900682,1188,1319,5379.167551
7,Qwen3-8B,olympiad,0.292285,197,674,5954.000000
8,Qwen3-8B,amc,0.625000,25,40,5920.675000
9,Qwen3-8B,aime-250,0.228000,57,250,6000.840000


In [4]:
for model in models:
    model_data = summary[summary["model"] == model]
    if model_data.empty:
        print(f"\n{model}: No results found\n")
        continue
    print(f"\n{'='*70}")
    print(f"Model: {model}")
    print(f"{'='*70}")
    print(f"{'Dataset':<15} {'Accuracy':>12} {'Correct':>10} {'Total':>8} {'Avg Tokens':>12}")
    print(f"{'-'*70}")
    for _, row in model_data.iterrows():
        print(f"{row['dataset']:<15} {row['accuracy']*100:>11.2f}% {row['num_correct']:>10} {row['num_total']:>8} {row['avg_tokens']:>12.1f}")
    total_correct = model_data["num_correct"].sum()
    total_problems = model_data["num_total"].sum()
    total_tokens = (model_data["avg_tokens"] * model_data["num_total"]).sum()
    overall_acc = total_correct / total_problems if total_problems > 0 else 0
    overall_avg = total_tokens / total_problems if total_problems > 0 else 0
    print(f"{'-'*70}")
    print(f"{'OVERALL':<15} {overall_acc*100:>11.2f}% {total_correct:>10} {total_problems:>8} {overall_avg:>12.1f}")
    print(f"{'='*70}")


Model: L1-Qwen3-8B-Max
Dataset             Accuracy    Correct    Total   Avg Tokens
----------------------------------------------------------------------
math-500              73.15%        365      499       2429.8
gsm8k                 80.59%       1063     1319       2137.8
olympiad              34.42%        232      674       3119.3
amc                   72.50%         29       40       2749.2
aime-250              38.00%         95      250       3406.5
----------------------------------------------------------------------
OVERALL               64.13%       1784     2782       2550.8

Model: Qwen3-8B
Dataset             Accuracy    Correct    Total   Avg Tokens
----------------------------------------------------------------------
math-500              75.75%        378      499       5895.4
gsm8k                 90.07%       1188     1319       5379.2
olympiad              29.23%        197      674       5954.0
amc                   62.50%         25       40       5920.7
ai

In [5]:

# Load all AIME and Olympiad wrong answers across all models
target_datasets = ["olympiad"]

wrong_answers = []
for model in models:
    model_dir = results_path / model
    for dataset in target_datasets:
        parquet_file = model_dir / f"{dataset}_results.parquet"
        if not parquet_file.exists():
            continue
        df = pd.read_parquet(parquet_file)
        wrong = df[df["is_correct"] == False].copy()
        wrong["model"] = model
        wrong["dataset"] = dataset
        wrong_answers.append(wrong)

wrong_df = pd.concat(wrong_answers, ignore_index=True)

# Display summary of wrong answers per model/dataset
print("Wrong answers per model/dataset:")
print(wrong_df.groupby(["model", "dataset"]).size().unstack(fill_value=0).to_string())
print(f"\nTotal wrong answers: {len(wrong_df)}")



Wrong answers per model/dataset:
dataset                    olympiad
model                              
L1-Qwen3-8B-Max                 442
LCR1_7B                         414
QwQ-32B-thinkprune-iter2k       378
Qwen3-8B                        477
TokenSkip-Qwen2                 488

Total wrong answers: 2199


In [18]:
from eval_pipeline import extract_boxed

ModuleNotFoundError: No module named 'vllm'